# FineSight-Reasoning Dataset Generation

**Dataset Two: Fine-Grained Visual Reasoning Evaluation Set**

Goal: evaluate whether a VLM can reason about objects it perceives — counting, comparing sizes, ordering spatially, and handling visual interference.

| Task | Description | Sizes | Counts | Samples | Total |
|------|-------------|-------|--------|---------|-------|
| Spatial Chain Reasoning | List objects + color from left→right or top→bottom | 7 sizes | 2,4,8,10 | 20 | **560** |
| Size Comparison Chain Reasoning | List objects + color from smallest→largest or largest→smallest | 7 sizes | 2,4,8,10 | 20 | **560** |
| Objects Counting Chain Reasoning | List count per type and total | 7 sizes | 2,4,8,10,15,20 | 20 | **840** |
| Blur Interference Chain Reasoning | Count objects per type/color + total on blurred background | 7 sizes | 2,4,8,10 | 20 | **560** |

**Total: 2 520 images**, canvas 448 × 448.

## 1. Import Required Libraries

In [1]:
import sys
import json
import random
import csv
import time
from pathlib import Path
from collections import defaultdict
from datetime import datetime
from typing import Any

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from PIL import Image

# Locate repo root via the installed package
import finesightbench as _fb
repo_root = Path(_fb.__file__).resolve().parent.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

# ── core drawing primitives ──────────────────────────────────────────────────
from finesightbench.core.canvas import create_canvas, create_textured_canvas
from finesightbench.core.colors import COLORS, TARGET_COLORS, apply_blur
from finesightbench.core.objects import (
    ANIMAL_TYPES, LETTERS, SHAPE_TYPES, TARGET_SIZES,
    draw_animal, draw_block, draw_color_block, draw_dot,
    draw_letter, draw_shape, random_position,
)

# ── reuse private helpers from the existing reasoning generator ───────────────
from finesightbench.reasoning.generator import (
    _place_non_overlapping,
    _draw_target,
    _random_order,
    _difficulty,
    _gen_chain,         # → Spatial Chain Reasoning
    _VALUE_POOL,
    _OBJ_TYPE_LABEL,
    _OBJ_TYPE_CHOICES_HINT,
    _TARGET_COLOR_CHOICES,
    _SHAPE_CHOICES,
    _LETTER_CHOICES,
    _ANIMAL_CHOICES,
    _COLOR_BLOCK_CHOICES,
)

print("Repo root     :", repo_root)
print("Target sizes  :", TARGET_SIZES)
print("Target colors :", TARGET_COLORS)

Repo root     : /home/snt/projects_lujun/FineSightBench
Target sizes  : [4, 8, 12, 16, 24, 32, 48]
Target colors : ['red', 'green', 'blue', 'yellow', 'cyan', 'magenta', 'orange', 'purple', 'pink', 'brown', 'black', 'gray']


## 2. Global Configuration

In [2]:
CANVAS_SIZE  = 448
PIXEL_SIZES  = [4, 8, 12, 16, 24, 32, 48]   # 7 difficulty levels
SEED         = 42
N_PER_CONFIG = 20                             # samples per (task, size, count) triple

# Object count configurations per task
SPATIAL_COUNTS    = [2, 4, 8, 10]            # 4 levels
COMPARISON_COUNTS = [2, 4, 8, 10]            # 4 levels
COUNTING_COUNTS   = [2, 4, 8, 10, 15, 20]   # 6 levels
BLUR_COUNTS       = [2, 4, 8, 10]            # 4 levels

# Expected totals
N_SPATIAL    = len(PIXEL_SIZES) * len(SPATIAL_COUNTS)    * N_PER_CONFIG  # 560
N_COMPARISON = len(PIXEL_SIZES) * len(COMPARISON_COUNTS) * N_PER_CONFIG  # 560
N_COUNTING   = len(PIXEL_SIZES) * len(COUNTING_COUNTS)   * N_PER_CONFIG  # 840
N_BLUR       = len(PIXEL_SIZES) * len(BLUR_COUNTS)       * N_PER_CONFIG  # 560
N_TOTAL      = N_SPATIAL + N_COMPARISON + N_COUNTING + N_BLUR            # 2520

OUTPUT_DIR   = repo_root / "data" / "full_reasoning"

print(f"Canvas size         : {CANVAS_SIZE} × {CANVAS_SIZE} px")
print(f"Pixel sizes         : {PIXEL_SIZES}")
print(f"Samples per config  : {N_PER_CONFIG}")
print()
print(f"Spatial counts      : {SPATIAL_COUNTS}  → {N_SPATIAL} images")
print(f"Comparison counts   : {COMPARISON_COUNTS} → {N_COMPARISON} images")
print(f"Counting counts     : {COUNTING_COUNTS} → {N_COUNTING} images")
print(f"Blur counts         : {BLUR_COUNTS}  → {N_BLUR} images")
print(f"────────────────────────────────────────────")
print(f"Total               :                  {N_TOTAL} images")
print(f"Output directory    : {OUTPUT_DIR}")

Canvas size         : 448 × 448 px
Pixel sizes         : [4, 8, 12, 16, 24, 32, 48]
Samples per config  : 20

Spatial counts      : [2, 4, 8, 10]  → 560 images
Comparison counts   : [2, 4, 8, 10] → 560 images
Counting counts     : [2, 4, 8, 10, 15, 20] → 840 images
Blur counts         : [2, 4, 8, 10]  → 560 images
────────────────────────────────────────────
Total               :                  2520 images
Output directory    : /home/snt/projects_lujun/FineSightBench/data/full_reasoning


## 3. New Generator Functions

Three new generator functions are defined here, complementing the existing `_gen_chain` (reused for Spatial Chain Reasoning):

| Function | Task |
|----------|------|
| `_gen_chain` (existing) | Spatial Chain Reasoning |
| `_gen_comparison_chain` | Size Comparison Chain Reasoning |
| `_gen_counting_chain` | Objects Counting Chain Reasoning |
| `_gen_blur_chain` | Blur Interference Chain Reasoning |

In [9]:

# ─────────────────────────────────────────────────────────────────────────────
# Task 2: Size Comparison Chain Reasoning
#
# n objects of the SAME type but DIFFERENT sizes and DIFFERENT colors.
# Sizes are sampled from PIXEL_SIZES (extended with midpoints for large n),
# with at least 2 PIXEL_SIZES index levels between consecutive sizes.
# Question: list them from smallest to largest (or largest to smallest) by size,
# each described as "<color> <identity>".
# Answer format: "<color1> <id1>, <color2> <id2>, ..."
# ─────────────────────────────────────────────────────────────────────────────
def _gen_comparison_chain(cw: int, ch: int, base_size: int, count: int = 4) -> dict[str, Any]:
    n = count
    obj_type = random.choice(["letter", "animal", "shape", "dot"])

    # ── Build extended size pool: PIXEL_SIZES + midpoints ───────────────────
    # e.g. [4, 6, 8, 10, 12, 14, 16, 20, 24, 28, 32, 40, 48]  (13 values)
    _PS = PIXEL_SIZES
    ext: list[int] = []
    for _i in range(len(_PS) - 1):
        ext.append(_PS[_i])
        ext.append((_PS[_i] + _PS[_i + 1]) // 2)
    ext.append(_PS[-1])

    # Anchor: index of base_size in ext (base_size is always in PIXEL_SIZES ⊂ ext)
    _base = ext.index(base_size) if base_size in ext else 0

    # Try progressively smaller gaps:
    #   gap=4  → skips 1 midpoint per step ≈ 2 PIXEL_SIZES levels apart
    #   gap=2  → 1 PIXEL_SIZES level apart
    #   gap=1  → consecutive ext entries (still distinct sizes)
    _chosen: list[int] = []
    for _gap in [4, 2, 1]:
        _pool = ext[_base::_gap]
        if len(_pool) >= n:
            _chosen = sorted(random.sample(_pool, n))
            break

    if not _chosen:
        # Fallback for very large n or high base_idx with few options above:
        # expand ext further until we have enough, then sample
        _pool = list(ext)
        while len(_pool) < n:
            _diffs = [(_pool[_i + 1] - _pool[_i], _i) for _i in range(len(_pool) - 1)]
            _mi = max(_diffs, key=lambda x: x[0])[1]
            _pool.insert(_mi + 1, (_pool[_mi] + _pool[_mi + 1]) // 2)
        _chosen = sorted(random.sample(_pool, n))

    sizes: list[int] = _chosen

    # ── Assign a distinct color to each object ───────────────────────────────
    color_pool = [c for c in TARGET_COLORS if c != "white"]
    random.shuffle(color_pool)
    colors = [color_pool[i % len(color_pool)] for i in range(n)]

    # Shuffle so draw order ≠ size order
    shuffled = list(zip(sizes, colors))
    random.shuffle(shuffled)
    shuffled_sizes, shuffled_colors = zip(*shuffled)

    positions = _place_non_overlapping(cw, ch, list(shuffled_sizes))

    img = create_canvas(cw, ch)
    targets = []
    for pos, sz, col in zip(positions, shuffled_sizes, shuffled_colors):
        val = random.choice(_VALUE_POOL[obj_type])
        color_rgb = COLORS[col]
        _draw_target(img, obj_type, pos, sz, color_rgb, val)
        targets.append({
            "type": obj_type, "value": val, "size": sz,
            "position": list(pos), "color": col, "color_rgb": list(color_rgb),
        })

    ascending = random.random() < 0.5
    sort_dir  = "smallest to largest" if ascending else "largest to smallest"
    sorted_t  = sorted(targets, key=lambda t: t["size"], reverse=not ascending)
    answer    = ", ".join(f"{t['color']} {t['value']}" for t in sorted_t)

    type_label = _OBJ_TYPE_LABEL[obj_type]
    hint       = _OBJ_TYPE_CHOICES_HINT.get(obj_type, "")
    question = (
        f"List all {type_label} in the image from {sort_dir} by size. "
        "Describe each as '<color> <identity>', separated by commas. "
        f"Colors are drawn from: {_TARGET_COLOR_CHOICES}.{hint}"
    )

    return {
        "image": img,
        "task_type": "comparison_chain",
        "question": question,
        "answer": answer,
        "sub_answers": {"direction": sort_dir, "order": [f"{t['color']} {t['value']}" for t in sorted_t]},
        "targets": targets,
    }


# ─────────────────────────────────────────────────────────────────────────────
# -----------------------------------------------------------------------------
# Task 3: Objects Counting Chain Reasoning
#
# Pick ONE category: "animal", "shape", or "color_block".
# Generate n_total objects, all of that category but with DIFFERENT specific
# identities (different animal species / shape types / colors).
# Question: give count per identity and total.
# Answer format: "<n1> <id1>, <n2> <id2>, ...; total: <N>"
# -----------------------------------------------------------------------------

def _gen_counting_chain(cw: int, ch: int, base_size: int, count: int = 6) -> dict[str, Any]:
    n_total  = count
    category = random.choice(["animal", "shape", "color_block"])

    if category == "animal":
        pool = list(ANIMAL_TYPES)
        n_ids = random.randint(2, min(len(pool), n_total, 6))
        chosen = random.sample(pool, n_ids)
        uni_color_name = random.choice([c for c in TARGET_COLORS if c not in ("white", "black")])
        uni_color_rgb  = COLORS[uni_color_name]
    elif category == "shape":
        pool = list(SHAPE_TYPES)
        n_ids = random.randint(2, min(len(pool), n_total, 6))
        chosen = random.sample(pool, n_ids)
        uni_color_name = random.choice([c for c in TARGET_COLORS if c not in ("white", "black")])
        uni_color_rgb  = COLORS[uni_color_name]
    else:  # color_block
        pool = [c for c in TARGET_COLORS if c not in ("black", "white", "gray")]
        n_ids = random.randint(2, min(len(pool), n_total, 6))
        chosen = random.sample(pool, n_ids)
        uni_color_name = None
        uni_color_rgb  = None

    # Distribute n_total among chosen identities (each gets at least 1)
    counts_per_id = [1] * n_ids
    for _ in range(n_total - n_ids):
        counts_per_id[random.randint(0, n_ids - 1)] += 1

    positions = _place_non_overlapping(cw, ch, [base_size] * n_total)
    img = create_canvas(cw, ch)
    targets: list[dict] = []
    pos_idx = 0
    for identity, cnt in zip(chosen, counts_per_id):
        for _ in range(cnt):
            if category == "animal":
                _draw_target(img, "animal", positions[pos_idx], base_size, uni_color_rgb, identity)
                targets.append({
                    "type": "animal", "value": identity, "size": base_size,
                    "position": list(positions[pos_idx]),
                    "color": uni_color_name, "color_rgb": list(uni_color_rgb),
                })
            elif category == "shape":
                _draw_target(img, "shape", positions[pos_idx], base_size, uni_color_rgb, identity)
                targets.append({
                    "type": "shape", "value": identity, "size": base_size,
                    "position": list(positions[pos_idx]),
                    "color": uni_color_name, "color_rgb": list(uni_color_rgb),
                })
            else:  # color_block
                color_rgb = COLORS[identity]
                draw_color_block(img, positions[pos_idx], base_size, color_rgb)
                targets.append({
                    "type": "color_block", "value": identity, "size": base_size,
                    "position": list(positions[pos_idx]),
                    "color": identity, "color_rgb": list(color_rgb),
                })
            pos_idx += 1

    count_parts   = [f"{cnt} {identity}" for identity, cnt in zip(chosen, counts_per_id)]
    answer        = ", ".join(count_parts) + f"; total: {n_total}"
    identity_list = ", ".join(chosen)

    if category == "animal":
        question = (
            "The image contains different animals. "
            "Count the number of each type of animal and give the total. "
            "Answer in the format: '<count> <animal>, ...; total: <N>'. "
            f"Animals present: {identity_list}."
        )
    elif category == "shape":
        question = (
            "The image contains different geometric shapes. "
            "Count the number of each shape type and give the total. "
            "Answer in the format: '<count> <shape>, ...; total: <N>'. "
            f"Shapes present: {identity_list}."
        )
    else:
        question = (
            "The image contains colored blocks of different colors. "
            "Count the number of blocks for each color and give the total. "
            "Answer in the format: '<count> <color>, ...; total: <N>'. "
            f"Colors present: {identity_list}."
        )

    return {
        "image": img,
        "task_type": "counting_chain",
        "question": question,
        "answer": answer,
        "sub_answers": {
            "category": category,
            "counts": dict(zip(chosen, counts_per_id)),
            "total": n_total,
        },
        "targets": targets,
    }


# -----------------------------------------------------------------------------
# Task 4: Blur Interference Chain Reasoning
#
# Blurred textured background; n objects drawn on top.
# Randomly chooses between two sub-modes:
#   "by_color"    -- colored blocks, different colors -> count per color + total
#   "by_identity" -- ONE category (animal/shape/color_block), different specific
#                    identities -> count per identity + total
# Answer format: "<n1> <label1>, <n2> <label2>, ...; total: <N>"
# -----------------------------------------------------------------------------
def _gen_blur_chain(cw: int, ch: int, base_size: int, count: int = 4) -> dict[str, Any]:
    blur_radius = random.choice([8.0, 12.0, 16.0, 20.0])
    bg = create_textured_canvas(cw, ch, density=0.05)
    bg = apply_blur(bg, blur_radius)

    n_total    = count
    mode       = random.choice(["by_color", "by_identity"])
    color_pool = [c for c in TARGET_COLORS if c not in ("black", "white", "gray")]

    positions = _place_non_overlapping(cw, ch, [base_size] * n_total, margin=20)
    targets: list[dict] = []

    if mode == "by_color":
        # Colored blocks with different colors; count per color
        obj_type = random.choice(["block", "color_block"])
        random.shuffle(color_pool)
        assigned_colors = [color_pool[i % len(color_pool)] for i in range(n_total)]
        for pos, col in zip(positions, assigned_colors):
            color_rgb = COLORS[col]
            draw_color_block(bg, pos, base_size, color_rgb)
            targets.append({
                "type": obj_type, "value": col, "size": base_size,
                "position": list(pos), "color": col, "color_rgb": list(color_rgb),
            })
        color_counts: dict[str, int] = defaultdict(int)
        for t in targets:
            color_counts[t["color"]] += 1
        count_parts = [f"{v} {k}" for k, v in sorted(color_counts.items())]
        answer     = ", ".join(count_parts) + f"; total: {n_total}"
        color_list = ", ".join(sorted(color_counts.keys()))
        question = (
            "The image shows colored blocks on a blurred, textured background. "
            "Count the number of blocks for each color and give the total. "
            "Answer in the format: '<count> <color>, ...; total: <N>'. "
            f"Colors present: {color_list}. "
            "(Ignore the background texture.)"
        )
        sub_answers = {"counts_by_color": dict(color_counts), "total": n_total}

    else:  # by_identity: one category, vary specific identities
        category = random.choice(["animal", "shape", "color_block"])
        if category == "animal":
            pool = list(ANIMAL_TYPES)
            n_ids = random.randint(2, min(len(pool), n_total, 6))
            chosen = random.sample(pool, n_ids)
            uni_color_name = random.choice([c for c in TARGET_COLORS if c not in ("white", "black")])
            uni_color_rgb  = COLORS[uni_color_name]
        elif category == "shape":
            pool = list(SHAPE_TYPES)
            n_ids = random.randint(2, min(len(pool), n_total, 6))
            chosen = random.sample(pool, n_ids)
            uni_color_name = random.choice([c for c in TARGET_COLORS if c not in ("white", "black")])
            uni_color_rgb  = COLORS[uni_color_name]
        else:  # color_block
            pool = [c for c in TARGET_COLORS if c not in ("black", "white", "gray")]
            n_ids = random.randint(2, min(len(pool), n_total, 6))
            chosen = random.sample(pool, n_ids)
            uni_color_name = None
            uni_color_rgb  = None

        counts_per_id = [1] * n_ids
        for _ in range(n_total - n_ids):
            counts_per_id[random.randint(0, n_ids - 1)] += 1

        pos_idx = 0
        for identity, cnt in zip(chosen, counts_per_id):
            for _ in range(cnt):
                if category == "animal":
                    _draw_target(bg, "animal", positions[pos_idx], base_size, uni_color_rgb, identity)
                    targets.append({
                        "type": "animal", "value": identity, "size": base_size,
                        "position": list(positions[pos_idx]),
                        "color": uni_color_name, "color_rgb": list(uni_color_rgb),
                    })
                elif category == "shape":
                    _draw_target(bg, "shape", positions[pos_idx], base_size, uni_color_rgb, identity)
                    targets.append({
                        "type": "shape", "value": identity, "size": base_size,
                        "position": list(positions[pos_idx]),
                        "color": uni_color_name, "color_rgb": list(uni_color_rgb),
                    })
                else:  # color_block
                    color_rgb = COLORS[identity]
                    draw_color_block(bg, positions[pos_idx], base_size, color_rgb)
                    targets.append({
                        "type": "color_block", "value": identity, "size": base_size,
                        "position": list(positions[pos_idx]),
                        "color": identity, "color_rgb": list(color_rgb),
                    })
                pos_idx += 1

        count_parts   = [f"{cnt} {identity}" for identity, cnt in zip(chosen, counts_per_id)]
        answer        = ", ".join(count_parts) + f"; total: {n_total}"
        identity_list = ", ".join(chosen)

        if category == "animal":
            question = (
                "The image shows animals on a blurred, textured background. "
                "Count the number of each type of animal and give the total. "
                "Answer in the format: '<count> <animal>, ...; total: <N>'. "
                f"Animals present: {identity_list}. "
                "(Ignore the background texture.)"
            )
        elif category == "shape":
            question = (
                "The image shows geometric shapes on a blurred, textured background. "
                "Count the number of each shape type and give the total. "
                "Answer in the format: '<count> <shape>, ...; total: <N>'. "
                f"Shapes present: {identity_list}. "
                "(Ignore the background texture.)"
            )
        else:
            question = (
                "The image shows colored blocks on a blurred, textured background. "
                "Count the number of blocks for each color and give the total. "
                "Answer in the format: '<count> <color>, ...; total: <N>'. "
                f"Colors present: {identity_list}. "
                "(Ignore the background texture.)"
            )
        sub_answers = {
            "category": category,
            "counts_by_identity": dict(zip(chosen, counts_per_id)),
            "total": n_total,
        }

    return {
        "image": bg,
        "task_type": "blur_chain",
        "question": question,
        "answer": answer,
        "sub_answers": sub_answers,
        "targets": targets,
        "extra": {"blur_radius": blur_radius, "mode": mode},
    }



print("Generator functions defined:")
print("  _gen_chain            → spatial_chain")
print("  _gen_comparison_chain → comparison_chain")
print("  _gen_counting_chain   → counting_chain")
print("  _gen_blur_chain       → blur_chain")


Generator functions defined:
  _gen_chain            → spatial_chain
  _gen_comparison_chain → comparison_chain
  _gen_counting_chain   → counting_chain
  _gen_blur_chain       → blur_chain


## 4. Generate the Full Reasoning Dataset

Iterates over all (task, pixel size, object count) triples and saves images + metadata.

> **Estimated time:** ~1–3 minutes.

In [10]:
random.seed(SEED)

img_dir = OUTPUT_DIR / "images"
img_dir.mkdir(parents=True, exist_ok=True)

# Task registry: (task_name, generator_fn, count_list)
TASK_REGISTRY = [
    ("spatial_chain",    _gen_chain,            SPATIAL_COUNTS),
    ("comparison_chain", _gen_comparison_chain,  COMPARISON_COUNTS),
    ("counting_chain",   _gen_counting_chain,    COUNTING_COUNTS),
    ("blur_chain",       _gen_blur_chain,        BLUR_COUNTS),
]

samples: list[dict[str, Any]] = []
idx = 0
t0  = time.time()

for task_name, gen_fn, count_list in TASK_REGISTRY:
    task_t0 = time.time()
    for size in PIXEL_SIZES:
        diff = _difficulty(size)
        for cnt in count_list:
            for _ in range(N_PER_CONFIG):
                result = gen_fn(CANVAS_SIZE, CANVAS_SIZE, size, count=cnt)
                image  = result.pop("image")

                image_id = f"reasoning_{task_name}_{size}px_n{cnt}_{idx:05d}"
                rel_path = f"images/{image_id}.png"
                image.save(img_dir / f"{image_id}.png")

                sample: dict[str, Any] = {
                    "image_id"        : image_id,
                    "image_path"      : rel_path,
                    "dataset"         : "reasoning",
                    "task_type"       : result["task_type"],
                    "question"        : result["question"],
                    "answer"          : result["answer"],
                    "difficulty"      : diff,
                    "generation_mode" : "difficulty_x_count",
                    "metadata": {
                        "canvas_size"      : [CANVAS_SIZE, CANVAS_SIZE],
                        "pixel_size"       : size,
                        "num_targets"      : cnt,
                        "background_color" : "white",
                        "targets"          : result["targets"],
                    },
                }
                if "sub_answers" in result:
                    sample["metadata"]["sub_answers"] = result["sub_answers"]
                if "extra" in result:
                    sample["metadata"]["extra"] = result["extra"]

                samples.append(sample)
                idx += 1

    task_elapsed = time.time() - task_t0
    n_task = sum(1 for s in samples if s["task_type"] == task_name
                 or s["task_type"] == {
                     "spatial_chain": "chain_reasoning",
                     "comparison_chain": "comparison_chain",
                     "counting_chain": "counting_chain",
                     "blur_chain": "blur_chain",
                 }[task_name])
    print(f"[{task_name:<22s}] done — {idx:>5d} total samples  ({task_elapsed:.1f}s)")

elapsed = time.time() - t0

# Write labels.json
dataset_meta = {
    "dataset_info": {
        "name"            : "FineSight-Reasoning",
        "version"         : "2.0",
        "description"     : "Fine-grained visual reasoning evaluation for VLMs (full dataset)",
        "creation_date"   : datetime.now().isoformat(),
        "num_samples"     : len(samples),
        "task_types"      : [t for t, _, _ in TASK_REGISTRY],
        "generation_mode" : "difficulty_x_count",
        "pixel_sizes"     : PIXEL_SIZES,
        "count_configs"   : {
            "spatial_chain"    : SPATIAL_COUNTS,
            "comparison_chain" : COMPARISON_COUNTS,
            "counting_chain"   : COUNTING_COUNTS,
            "blur_chain"       : BLUR_COUNTS,
        },
    },
    "samples": samples,
}

labels_path = OUTPUT_DIR / "labels.json"
labels_path.write_text(json.dumps(dataset_meta, indent=2, ensure_ascii=False))

print(f"\nDone in {elapsed:.1f}s  —  {len(samples)} total images")
print(f"Labels saved to: {labels_path}")

[spatial_chain         ] done —   560 total samples  (2.9s)
[comparison_chain      ] done —  1120 total samples  (2.9s)
[counting_chain        ] done —  1960 total samples  (4.1s)
[blur_chain            ] done —  2520 total samples  (22.7s)

Done in 32.6s  —  2520 total images
Labels saved to: /home/snt/projects_lujun/FineSightBench/data/full_reasoning/labels.json


## 5. Dataset Statistics

In [5]:
by_task:       dict[str, int]           = defaultdict(int)
by_size:       dict[int, int]           = defaultdict(int)
by_count:      dict[int, int]           = defaultdict(int)
by_difficulty: dict[str, int]           = defaultdict(int)
by_task_size:  dict[tuple, int]         = defaultdict(int)

for s in samples:
    task = s["task_type"]
    size = s["metadata"]["pixel_size"]
    cnt  = s["metadata"]["num_targets"]
    diff = s["difficulty"]
    by_task[task]            += 1
    by_size[size]            += 1
    by_count[cnt]            += 1
    by_difficulty[diff]      += 1
    by_task_size[(task, size)] += 1

print("=== Samples per Task ===")
for task, n in sorted(by_task.items()):
    print(f"  {task:<30s}: {n:>5d}")

print()
print("=== Samples per Pixel Size ===")
for size, n in sorted(by_size.items()):
    print(f"  {size:>3d} px : {n:>5d}")

print()
print("=== Samples per Object Count ===")
for cnt, n in sorted(by_count.items()):
    print(f"  count={cnt:>2d} : {n:>5d}")

print()
print("=== Difficulty Distribution ===")
for diff, n in sorted(by_difficulty.items()):
    print(f"  {diff:<10s}: {n:>5d}")

print()
print(f"Total samples : {len(samples)}")

=== Samples per Task ===
  blur_chain                    :   560
  chain_reasoning               :   560
  comparison_chain              :   560
  counting_chain                :   840

=== Samples per Pixel Size ===
    4 px :   360
    8 px :   360
   12 px :   360
   16 px :   360
   24 px :   360
   32 px :   360
   48 px :   360

=== Samples per Object Count ===
  count= 2 :   560
  count= 4 :   560
  count= 8 :   560
  count=10 :   560
  count=15 :   140
  count=20 :   140

=== Difficulty Distribution ===
  easy      :   720
  extreme   :   360
  hard      :   720
  medium    :   720

Total samples : 2520


## 6. Visualize Sample Images

Show a 4 × 7 grid: one representative sample per (task, pixel size) combination.

In [6]:
TASK_ORDER_VIS = [
    "chain_reasoning",   # spatial_chain uses existing task_type name
    "comparison_chain",
    "counting_chain",
    "blur_chain",
]
TASK_LABELS = {
    "chain_reasoning" : "spatial_chain",
    "comparison_chain": "comparison_chain",
    "counting_chain"  : "counting_chain",
    "blur_chain"      : "blur_chain",
}

# One representative per (task_type, size)
pool: dict[tuple, dict] = {}
for s in samples:
    key = (s["task_type"], s["metadata"]["pixel_size"])
    if key not in pool:
        pool[key] = s

rows = len(TASK_ORDER_VIS)
cols = len(PIXEL_SIZES)
fig, axes = plt.subplots(rows, cols, figsize=(cols * 2.2, rows * 2.5))
fig.suptitle("FineSight-Reasoning — one sample per (task, pixel size)", fontsize=13, y=1.01)

for r, task in enumerate(TASK_ORDER_VIS):
    for c, size in enumerate(PIXEL_SIZES):
        ax  = axes[r][c]
        key = (task, size)
        if key in pool:
            s   = pool[key]
            img = Image.open(OUTPUT_DIR / s["image_path"])
            ax.imshow(img)
            ax.set_title(f"{size}px", fontsize=8)
        else:
            ax.axis("off")
        if c == 0:
            ax.set_ylabel(TASK_LABELS.get(task, task), fontsize=7, rotation=0,
                          labelpad=110, ha="left", va="center")
        ax.set_xticks([])
        ax.set_yticks([])

plt.tight_layout()
vis_path = OUTPUT_DIR / "preview_grid.png"
plt.savefig(vis_path, dpi=100, bbox_inches="tight")
plt.show()
print(f"Saved preview grid → {vis_path}")

Saved preview grid → /home/snt/projects_lujun/FineSightBench/data/full_reasoning/preview_grid.png


## 7. Inspect Sample Q&A Pairs

Print representative question / answer pairs to verify the prompt format is correct.

In [11]:
seen: set[str] = set()
for s in samples:
    task = s["task_type"]
    size = s["metadata"]["pixel_size"]
    cnt  = s["metadata"]["num_targets"]
    key  = f"{task}|{size}|{cnt}"
    # Show one example per task at (12px, count=4) or first available combo
    if task not in seen and size == 12 and cnt in (4, 2):
        seen.add(task)
        print(f"{'='*70}")
        print(f"Task      : {task}")
        print(f"Size / N  : {size}px / {cnt} objects")
        print(f"Question  : {s['question'][:200]}")
        print(f"Answer    : {s['answer']}")
        if "sub_answers" in s.get("metadata", {}):
            print(f"Sub-ans   : {s['metadata']['sub_answers']}")

Task      : chain_reasoning
Size / N  : 12px / 2 objects
Question  : List all objects in the image from left to right. Describe each by its color and identity, in the form '<color> <identity>', separated by commas. Colors are drawn from: red, green, blue, yellow, cyan,
Answer    : red Q, purple A
Task      : comparison_chain
Size / N  : 12px / 2 objects
Question  : List all animals in the image from largest to smallest by size. Describe each as '<color> <identity>', separated by commas. Colors are drawn from: red, green, blue, yellow, cyan, magenta, orange, purp
Answer    : pink rabbit, magenta dog
Sub-ans   : {'direction': 'largest to smallest', 'order': ['pink rabbit', 'magenta dog']}
Task      : counting_chain
Size / N  : 12px / 2 objects
Question  : The image contains colored blocks of different colors. Count the number of blocks for each color and give the total. Answer in the format: '<count> <color>, ...; total: <N>'. Colors present: blue, cya
Answer    : 1 blue, 1 cyan; total: 

## 8. Summary Table and Export to CSV

In [12]:
DISPLAY_TASK_ORDER = [
    "chain_reasoning",
    "comparison_chain",
    "counting_chain",
    "blur_chain",
]

# Per-task totals across all sizes and counts
task_totals = {t: by_task.get(t, 0) for t in DISPLAY_TASK_ORDER}

print("=== Summary Table ===")
print(f"{'Task':<22s}", end="")
for sz in PIXEL_SIZES:
    print(f"  {sz:>3}px", end="")
print("   Total")
print("-" * (22 + len(PIXEL_SIZES) * 6 + 9))
for task in DISPLAY_TASK_ORDER:
    print(f"{task:<22s}", end="")
    row_total = 0
    for sz in PIXEL_SIZES:
        n = by_task_size.get((task, sz), 0)
        print(f"  {n:>4d}", end="")
        row_total += n
    print(f"  {row_total:>6d}")
print("-" * (22 + len(PIXEL_SIZES) * 6 + 9))
print(f"{'Total':<22s}", end="")
col_totals = [sum(by_task_size.get((t, sz), 0) for t in DISPLAY_TASK_ORDER) for sz in PIXEL_SIZES]
for ct in col_totals:
    print(f"  {ct:>4d}", end="")
print(f"  {sum(col_totals):>6d}")

# ── Export CSV ────────────────────────────────────────────────────────────────
csv_path   = OUTPUT_DIR / "metadata.csv"
fieldnames = [
    "image_id", "image_path", "task_type",
    "pixel_size", "num_targets", "difficulty",
    "question", "answer",
]

with open(csv_path, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    for s in samples:
        writer.writerow({
            "image_id"   : s["image_id"],
            "image_path" : s["image_path"],
            "task_type"  : s["task_type"],
            "pixel_size" : s["metadata"]["pixel_size"],
            "num_targets": s["metadata"]["num_targets"],
            "difficulty" : s["difficulty"],
            "question"   : s["question"],
            "answer"     : s["answer"],
        })

print(f"\nCSV written : {csv_path}  ({len(samples)} rows)")
print(f"JSON written: {labels_path}")

=== Summary Table ===
Task                      4px    8px   12px   16px   24px   32px   48px   Total
-------------------------------------------------------------------------
chain_reasoning           80    80    80    80    80    80    80     560
comparison_chain          80    80    80    80    80    80    80     560
counting_chain           120   120   120   120   120   120   120     840
blur_chain                80    80    80    80    80    80    80     560
-------------------------------------------------------------------------
Total                    360   360   360   360   360   360   360    2520

CSV written : /home/snt/projects_lujun/FineSightBench/data/full_reasoning/metadata.csv  (2520 rows)
JSON written: /home/snt/projects_lujun/FineSightBench/data/full_reasoning/labels.json


## 9. Re-label: JSON-format Q&A + JSONL Export

For each sample:
- **Question** is updated to instruct the VLM to reply with a strict JSON object.
- **Answer** becomes a key-value `dict` for field-by-field evaluation.

Output file: `labels.jsonl` — one JSON object per line, fully self-contained.

| Task | JSON answer schema |
|------|-----------------|
| `chain_reasoning` (spatial) | `{"objects": ["red A", "blue cat", ...]}` |
| `comparison_chain` | `{"objects": ["red A", "blue cat", ...]}` (sorted by size) |
| `counting_chain` (animal) | `{"counts": {"cat": 2, "dog": 1}, "total": 3}` |
| `counting_chain` (shape) | `{"counts": {"circle": 2, "star": 1}, "total": 3}` |
| `counting_chain` (color_block) | `{"counts": {"red": 2, "blue": 1}, "total": 3}` |
| `blur_chain` (by_color) | `{"counts": {"red": 2, "blue": 1}, "total": 3}` |
| `blur_chain` (by_identity) | `{"counts": {"cat": 2, "dog": 1}, "total": 3}` |

In [13]:
import json as _json
import re as _re
import finesightbench as _fb
from pathlib import Path
from finesightbench.core.objects import LETTERS, ANIMAL_TYPES, SHAPE_TYPES
from finesightbench.core.colors import TARGET_COLORS

# -- vocab strings -------------------------------------------------------
_TARGET_COLOR_CHOICES = ", ".join(TARGET_COLORS)
_LETTER_CHOICES       = ", ".join(LETTERS)
_ANIMAL_CHOICES       = ", ".join(ANIMAL_TYPES)
_SHAPE_CHOICES        = ", ".join(SHAPE_TYPES)

_OBJ_LABEL = {
    "letter": "letters", "animal": "animals", "block": "blocks",
    "color_block": "colored blocks", "shape": "shapes", "dot": "dots",
}
_OBJ_HINT = {
    "letter":     f" Identities are uppercase letters from: {', '.join(LETTERS)}.",
    "animal":     f" Identities are animals from: {', '.join(ANIMAL_TYPES)}.",
    "shape":      f" Identities are shapes from: {', '.join(SHAPE_TYPES)}.",
    "block":      "",
    "color_block": "",
    "dot":        "",
}

_REAS_OUTPUT_DIR = Path(_fb.__file__).resolve().parent.parent / "data" / "full_reasoning"


def _parse_count_answer(answer: str) -> tuple[dict, int]:
    """Parse 'N label1, N label2; total: N' into (counts_dict, total)."""
    body, total_part = answer.split(";")
    total  = int(total_part.strip().replace("total:", "").strip())
    counts = {}
    for part in body.split(","):
        part  = part.strip()
        n_str, label = part.split(" ", 1)
        counts[label.strip()] = int(n_str.strip())
    return counts, total


def _reasoning_qa(s: dict) -> tuple[str, dict]:
    """Return (new_question, answer_dict) for a reasoning sample."""
    task = s["task_type"]
    meta = s["metadata"]

    if task == "chain_reasoning":
        m         = _re.search(r"from (left to right|top to bottom)", s["question"])
        direction = m.group(0) if m else "from left to right"
        obj_type  = meta["targets"][0]["type"]
        type_lbl  = _OBJ_LABEL.get(obj_type, "objects")
        hint      = _OBJ_HINT.get(obj_type, "")
        objects    = [o.strip() for o in s["answer"].split(",")]
        answer_dict = {"objects": objects}
        q = (
            f"List all {type_lbl} in the image {direction}. "
            "Describe each by its color and identity as '<color> <identity>'. "
            '{"objects": ["<color> <identity>", ...]}. '
            f"Colors are from: {_TARGET_COLOR_CHOICES}.{hint}"
        )

    elif task == "comparison_chain":
        m         = _re.search(r"from (smallest to largest|largest to smallest)", s["question"])
        direction = m.group(0) if m else "from smallest to largest"
        obj_type  = meta["targets"][0]["type"]
        type_lbl  = _OBJ_LABEL.get(obj_type, "objects")
        hint      = _OBJ_HINT.get(obj_type, "")
        objects    = [o.strip() for o in s["answer"].split(",")]
        answer_dict = {"objects": objects}
        q = (
            f"List all {type_lbl} in the image {direction} by size. "
            "Describe each as '<color> <identity>'. "
            '{"objects": ["<color> <identity>", ...]}. '
            f"Colors are from: {_TARGET_COLOR_CHOICES}.{hint}"
        )

    elif task == "counting_chain":
        counts, total = _parse_count_answer(s["answer"])
        answer_dict   = {"counts": counts, "total": total}
        identity_list = ", ".join(counts.keys())
        tgt_type = meta["targets"][0]["type"] if meta["targets"] else "color_block"
        if tgt_type == "animal":
            q = (
                "The image contains different animals. "
                "Count the number of each type of animal and provide the total. "
                'Answer in JSON format only: {"counts": {"<animal>": <n>, ...}, "total": <N>}. '
                f"Animals present: {identity_list}. "
                f"Animal types are from: {_ANIMAL_CHOICES}."
            )
        elif tgt_type == "shape":
            q = (
                "The image contains different geometric shapes. "
                "Count the number of each shape type and provide the total. "
                'Answer in JSON format only: {"counts": {"<shape>": <n>, ...}, "total": <N>}. '
                f"Shapes present: {identity_list}. "
                f"Shape types are from: {_SHAPE_CHOICES}."
            )
        else:  # color_block
            q = (
                "The image contains colored blocks of different colors. "
                "Count the number of blocks for each color and provide the total. "
                'Answer in JSON format only: {"counts": {"<color>": <n>, ...}, "total": <N>}. '
                f"Colors present: {identity_list}. "
                f"Colors are from: {_TARGET_COLOR_CHOICES}."
            )

    elif task == "blur_chain":
        counts, total = _parse_count_answer(s["answer"])
        answer_dict   = {"counts": counts, "total": total}
        key_list      = ", ".join(counts.keys())
        mode          = meta.get("extra", {}).get("mode", "by_identity")
        if mode == "by_color":
            q = (
                "The image shows colored blocks on a blurred, textured background. "
                "Count the number of blocks for each color and provide the total. "
                'Answer in JSON format only: {"counts": {"<color>": <n>, ...}, "total": <N>}. '
                f"Colors present: {key_list}. (Ignore the background texture.)"
            )
        else:  # by_identity
            tgt_type = meta["targets"][0]["type"] if meta["targets"] else "color_block"
            if tgt_type == "animal":
                q = (
                    "The image shows animals on a blurred, textured background. "
                    "Count the number of each type of animal and provide the total. "
                    'Answer in JSON format only: {"counts": {"<animal>": <n>, ...}, "total": <N>}. '
                    f"Animals present: {key_list}. "
                    f"Animal types are from: {_ANIMAL_CHOICES}. (Ignore the background texture.)"
                )
            elif tgt_type == "shape":
                q = (
                    "The image shows geometric shapes on a blurred, textured background. "
                    "Count the number of each shape type and provide the total. "
                    'Answer in JSON format only: {"counts": {"<shape>": <n>, ...}, "total": <N>}. '
                    f"Shapes present: {key_list}. "
                    f"Shape types are from: {_SHAPE_CHOICES}. (Ignore the background texture.)"
                )
            else:  # color_block
                q = (
                    "The image shows colored blocks on a blurred, textured background. "
                    "Count the number of blocks for each color and provide the total. "
                    'Answer in JSON format only: {"counts": {"<color>": <n>, ...}, "total": <N>}. '
                    f"Colors present: {key_list}. "
                    f"Colors are from: {_TARGET_COLOR_CHOICES}. (Ignore the background texture.)"
                )

    else:
        q           = s["question"]
        answer_dict = {"value": s["answer"]}

    return q, answer_dict


# -- load existing labels -----------------------------------------------
with open(_REAS_OUTPUT_DIR / "labels.json", encoding="utf-8") as _f:
    _data = _json.load(_f)
_samples = _data["samples"]

# -- write JSONL --------------------------------------------------------
jsonl_path = _REAS_OUTPUT_DIR / "labels.jsonl"
with open(jsonl_path, "w", encoding="utf-8") as _f:
    for s in _samples:
        new_q, new_a = _reasoning_qa(s)
        row = dict(s)
        row["question"]      = new_q
        row["answer"]        = new_a
        row["answer_format"] = "json"
        _f.write(_json.dumps(row, ensure_ascii=False) + "\n")

print(f"Written {len(_samples)} lines -> {jsonl_path}")

# -- spot-check one sample per task ------------------------------------
_seen_r: set[str] = set()
for s in _samples:
    task = s["task_type"]
    if task not in _seen_r:
        _seen_r.add(task)
        new_q, new_a = _reasoning_qa(s)
        print(f"\n[{task}]")
        print(f"  Q : {new_q[:120]}")
        print(f"  A : {new_a}")

Written 2520 lines -> /home/snt/projects_lujun/FineSightBench/data/full_reasoning/labels.jsonl

[chain_reasoning]
  Q : List all dots in the image from top to bottom. Describe each by its color and identity as '<color> <identity>'. {"object
  A : {'objects': ['yellow dot', 'blue dot']}

[comparison_chain]
  Q : List all animals in the image from smallest to largest by size. Describe each as '<color> <identity>'. {"objects": ["<co
  A : {'objects': ['blue dog', 'magenta bird']}

[counting_chain]
  Q : The image contains colored blocks of different colors. Count the number of blocks for each color and provide the total. 
  A : {'counts': {'purple': 1, 'green': 1}, 'total': 2}

[blur_chain]
  Q : The image shows geometric shapes on a blurred, textured background. Count the number of each shape type and provide the 
  A : {'counts': {'hexagon': 1, 'square': 1}, 'total': 2}
